# Continuous VP Diffusion

This notebook uses the real [`ContinuousVPDiffusion`](./continuous.py) class together with the real `CSPTask` dataloader.

The goal is to make the training idea easy to follow:

1. Start from clean data `x0`
2. Sample a random diffusion time `t`
3. Create a noisy version `x_t`
4. Compute the exact target score
5. Train a model to predict that target

## Theory

For the VP forward process we use the closed-form Gaussian marginal

$$
x_t \mid x_0 \sim \mathcal{N}(\alpha(t)x_0, \sigma(t)^2 I)
$$

so we can sample directly with

$$
x_t = \alpha(t)x_0 + \sigma(t)\epsilon, \quad \epsilon \sim \mathcal{N}(0, I)
$$

and the exact conditional score target is

$$
\nabla_{x_t} \log p(x_t \mid x_0) = -\epsilon / \sigma(t)
$$

That means training becomes ordinary supervised learning on noisy inputs.

In [83]:
from pathlib import Path

import torch
from torch import nn

from kldm.data import CSPTask, DNGTask, resolve_data_root
from kldm.diffusionModels.continuous import ContinuousVPDiffusion
from kldm.scoreNetwork.scoreNetwork import CSPVNet

torch.manual_seed(7)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
root = resolve_data_root()

diffusion = ContinuousVPDiffusion().to(device)
diffusion

ContinuousVPDiffusion()

## 1. Load A Real CSP Batch

We use the real MatterGen-backed CSP loader.

- `CSPTask` gives a graph-level lattice representation in `batch.l`

In [84]:
csp_batch = next(iter(CSPTask().dataloader(root=root, split='train', batch_size=2, shuffle=False, download=True)))
csp_batch = csp_batch.to(device)

print('CSP lattice shape:', tuple(csp_batch.l.shape))
print('CSP atom shape   :', tuple(csp_batch.h.shape))

CSP lattice shape: (2, 6)
CSP atom shape   : (25,)


## 2. CSP Case: Diffuse The Lattice

For CSP we treat the lattice vector

$$
l = [\log a, \log b, \log c, \tan(\alpha - \pi/2), \tan(\beta - \pi/2), \tan(\gamma - \pi/2)]
$$

as the clean Euclidean variable `x0`.

In [85]:
csp_x0 = csp_batch.l
csp_t = torch.rand(csp_x0.shape[0], device=device)
csp_xt, csp_eps = diffusion.forward_sample(csp_t, csp_x0)
csp_target = diffusion.score_target(csp_t, csp_eps)

print('csp_t:', csp_t)
print('\nBefore diffusion (first structure):')
print(csp_x0[0])
print('\nNoise used:')
print(csp_eps[0])
print('\nAfter diffusion:')
print(csp_xt[0])
print('\nTarget score:')
print(csp_target[0])

csp_t: tensor([0.6592, 0.6569])

Before diffusion (first structure):
tensor([1.1083, 1.7294, 2.0766, 0.3155, 0.1934, 0.0000])

Noise used:
tensor([ 0.9468, -1.1143,  1.6908, -0.8948, -0.3556,  1.2324])

After diffusion:
tensor([ 1.3836, -0.0592,  2.5212, -0.6026, -0.2043,  1.0547])

Target score:
tensor([-1.1063,  1.3021, -1.9756,  1.0456,  0.4155, -1.4400])


## 3. Reshape The Lattice For A Tiny U-Net

Yang Song's tutorial uses a U-Net on images. Our lattice is only 6 numbers, so we pad it into a tiny `4 x 4` image.

This does **not** change the diffusion idea. It is only a convenient way to let us use a convolutional score network on lattice vectors.

In [86]:
def lattice_to_image(lattice: torch.Tensor) -> torch.Tensor:
    flat = torch.zeros(lattice.shape[0], 16, device=lattice.device, dtype=lattice.dtype)
    flat[:, :6] = lattice
    return flat.view(-1, 1, 4, 4)


def image_to_lattice(image: torch.Tensor) -> torch.Tensor:
    return image.view(image.shape[0], -1)[:, :6]


csp_x0_img = lattice_to_image(csp_batch.l)
print('lattice image shape:', tuple(csp_x0_img.shape))
print('\nOriginal lattice vector:')
print(csp_batch.l[0])
print('\nAs tiny image:')
print(csp_x0_img[0, 0])
print('\nRecovered lattice vector:')
print(image_to_lattice(csp_x0_img)[0])

lattice image shape: (2, 1, 4, 4)

Original lattice vector:
tensor([1.1083, 1.7294, 2.0766, 0.3155, 0.1934, 0.0000])

As tiny image:
tensor([[1.1083, 1.7294, 2.0766, 0.3155],
        [0.1934, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000]])

Recovered lattice vector:
tensor([1.1083, 1.7294, 2.0766, 0.3155, 0.1934, 0.0000])


## 4. A Small Time-Dependent U-Net Score Model

Now we switch from the simple vector MLP to a small U-Net-style score model.

This follows Yang Song's tutorial idea:

- encode time with Gaussian Fourier features
- inject time information at each stage
- predict a score with the same shape as the noisy input
- divide the output by the marginal standard deviation

In [87]:
class GaussianFourierProjection(nn.Module):
    """Gaussian random features for encoding time steps."""

    def __init__(self, embed_dim: int, scale: float = 30.0) -> None:
        super().__init__()
        self.W = nn.Parameter(torch.randn(embed_dim // 2) * scale, requires_grad=False)

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        t_proj = t[:, None] * self.W[None, :] * 2 * torch.pi
        return torch.cat([torch.sin(t_proj), torch.cos(t_proj)], dim=-1)


class Dense(nn.Module):
    """A fully connected layer that reshapes outputs to feature maps."""

    def __init__(self, input_dim: int, output_dim: int) -> None:
        super().__init__()
        self.dense = nn.Linear(input_dim, output_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dense(x)[..., None, None]


class ScoreNet(nn.Module):
    """A small time-dependent U-Net for lattice images."""

    def __init__(self, marginal_prob_std, channels=(32, 64, 128), embed_dim: int = 256) -> None:
        super().__init__()
        self.embed = nn.Sequential(
            GaussianFourierProjection(embed_dim=embed_dim),
            nn.Linear(embed_dim, embed_dim),
        )
        self.conv1 = nn.Conv2d(1, channels[0], 3, padding=1, bias=False)
        self.dense1 = Dense(embed_dim, channels[0])
        self.gnorm1 = nn.GroupNorm(4, num_channels=channels[0])
        self.conv2 = nn.Conv2d(channels[0], channels[1], 3, stride=2, padding=1, bias=False)
        self.dense2 = Dense(embed_dim, channels[1])
        self.gnorm2 = nn.GroupNorm(8, num_channels=channels[1])
        self.conv3 = nn.Conv2d(channels[1], channels[2], 3, stride=2, padding=1, bias=False)
        self.dense3 = Dense(embed_dim, channels[2])
        self.gnorm3 = nn.GroupNorm(8, num_channels=channels[2])

        self.tconv2 = nn.ConvTranspose2d(channels[2], channels[1], 4, stride=2, padding=1, bias=False)
        self.dense4 = Dense(embed_dim, channels[1])
        self.tgnorm2 = nn.GroupNorm(8, num_channels=channels[1])
        self.tconv1 = nn.ConvTranspose2d(channels[1] + channels[1], channels[0], 4, stride=2, padding=1, bias=False)
        self.dense5 = Dense(embed_dim, channels[0])
        self.tgnorm1 = nn.GroupNorm(4, num_channels=channels[0])
        self.out = nn.Conv2d(channels[0] + channels[0], 1, 3, padding=1)

        self.act = lambda x: x * torch.sigmoid(x)
        self.marginal_prob_std = marginal_prob_std

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        embed = self.act(self.embed(t))

        h1 = self.conv1(x)
        h1 = h1 + self.dense1(embed)
        h1 = self.gnorm1(h1)
        h1 = self.act(h1)

        h2 = self.conv2(h1)
        h2 = h2 + self.dense2(embed)
        h2 = self.gnorm2(h2)
        h2 = self.act(h2)

        h3 = self.conv3(h2)
        h3 = h3 + self.dense3(embed)
        h3 = self.gnorm3(h3)
        h3 = self.act(h3)

        h = self.tconv2(h3)
        h = h + self.dense4(embed)
        h = self.tgnorm2(h)
        h = self.act(h)

        h = self.tconv1(torch.cat([h, h2], dim=1))
        h = h + self.dense5(embed)
        h = self.tgnorm1(h)
        h = self.act(h)

        h = self.out(torch.cat([h, h1], dim=1))
        h = h / self.marginal_prob_std(t)[:, None, None, None]
        return h


def denoise_score_matching_loss(
    pred: torch.Tensor,
    target: torch.Tensor,
    t: torch.Tensor,
    diffusion: ContinuousVPDiffusion,
    lambda_weight=None,
) -> torch.Tensor:
    """Small denoising score matching loss in the paper form.

    We use the DSM objective

        lambda(t) * || s_theta(x_t, t) - grad log p_{t|0}(x_t | x_0) ||^2

    where grad log p_{t|0}(x_t | x_0) is available in closed form for the
    Gaussian VP forward process.
    """
    if lambda_weight is None:
        lambda_weight = diffusion.sigma(t) ** 2
    weight = diffusion._match_dims(lambda_weight, pred)
    return (weight * (pred - target) ** 2).mean()


lattice_model = ScoreNet(marginal_prob_std=diffusion.sigma).to(device)
optimizer = torch.optim.Adam(lattice_model.parameters(), lr=1e-3)
lattice_model

ScoreNet(
  (embed): Sequential(
    (0): GaussianFourierProjection()
    (1): Linear(in_features=256, out_features=256, bias=True)
  )
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (dense1): Dense(
    (dense): Linear(in_features=256, out_features=32, bias=True)
  )
  (gnorm1): GroupNorm(4, 32, eps=1e-05, affine=True)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (dense2): Dense(
    (dense): Linear(in_features=256, out_features=64, bias=True)
  )
  (gnorm2): GroupNorm(8, 64, eps=1e-05, affine=True)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (dense3): Dense(
    (dense): Linear(in_features=256, out_features=128, bias=True)
  )
  (gnorm3): GroupNorm(8, 128, eps=1e-05, affine=True)
  (tconv2): ConvTranspose2d(128, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
  (dense4): Dense(
    (dense): Linear(in_features=256, out_features=64,

## 5. One Training Step

This is the core diffusion training pattern for the lattice case.

For this notebook we use the paper-style **denoising score matching** loss:

$$
\mathcal{L}_{DSM} = \mathbb{E}\left[ \lambda(t) \left\| s_\theta(x_t, t) - \nabla_{x_t} \log p_{t|0}(x_t \mid x_0) \right\|_2^2 \right]
$$

For the VP Gaussian forward kernel we know the target exactly:

$$
s^*(x_t, t \mid x_0) = -\epsilon / \sigma(t)
$$

In this notebook we choose the simple weighting

$$
\lambda(t) = \sigma(t)^2
$$

which keeps the implementation small and matches the weighted DSM form you quoted.

Step by step:

1. sample clean lattice data `x0`
2. reshape it into a tiny image
3. sample random time `t`
4. build noisy `x_t`
5. compute the exact target score
6. train the U-Net to predict that score

In [47]:
batch = next(iter(CSPTask().dataloader(root=root, split='train', batch_size=8, shuffle=True, download=True))).to(device)

x0 = lattice_to_image(batch.l)
t = torch.rand(x0.shape[0], device=x0.device)
x_t, eps = diffusion.forward_sample(t, x0)
target = diffusion.score_target(t, eps)

pred = lattice_model(x_t, t)
loss = denoise_score_matching_loss(pred, target, t, diffusion)

optimizer.zero_grad()
loss.backward()
optimizer.step()

print('x0 image shape :', tuple(x0.shape))
print('x_t shape      :', tuple(x_t.shape))
print('target shape   :', tuple(target.shape))
print('pred shape     :', tuple(pred.shape))
print('loss        :', float(loss))

x0 shape    : (8, 6)
x_t shape   : (8, 6)
target shape: (8, 6)
pred shape  : (8, 6)
loss        : 1.0432778596878052


## 6. The Training Idea In Loop Form

This is the same idea written as a small loop.

We are **not** simulating many small forward steps for each sample. Instead, for each batch we directly sample a random time `t` and jump to the corresponding noisy point `x_t` using the closed-form formula.

In [ ]:
train_loader = CSPTask().dataloader(root=root, split='train', batch_size=64, shuffle=False, download=True)
num_epochs = 1000
print_every = 200

for epoch in range(num_epochs):
    running_loss = 0.0

    for step, batch in enumerate(train_loader):
        batch = batch.to(device)

        x0 = lattice_to_image(batch.l)
        t = torch.rand(x0.shape[0], device=x0.device)
        x_t, eps = diffusion.forward_sample(t, x0)
        target = diffusion.score_target(t, eps)

        pred = lattice_model(x_t, t)
        loss = denoise_score_matching_loss(pred, target, t, diffusion)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += float(loss)

        if step % print_every == 0:
            print(f'epoch={epoch:03d} step={step:03d} loss={float(loss):.6f}')

    epoch_loss = running_loss / (step + 1)
    print(f'epoch={epoch:03d} mean_loss={epoch_loss:.6f}')

epoch=000 step=000 loss=0.939717
epoch=000 step=200 loss=0.152089


## 7. Use The Real KLDM Score Network

Now we switch from the toy lattice U-Net to the real [`CSPVNet`](../scoreNetwork/scoreNetwork.py).

Here we use a **simplified Algorithm 1 / 2** version:

- load a `DNGTask` batch so we have one-hot atoms
- diffuse the lattice `l`
- diffuse the atom features `h`
- keep the fractional coordinates `pos` fixed
- keep `v = 0` instead of diffusing velocity
- pass `(t, pos, v, h, l, node_index, edge_node_index)` into `CSPVNet`

So this section intentionally **forgets velocity diffusion and fractional-coordinate diffusion** and focuses only on lattice and atom targets.

In [88]:
dng_batch = next(iter(DNGTask().dataloader(root=root, split='train', batch_size=4, shuffle=False, download=True))).to(device)

# Sample one graph-time per crystal, then reuse it for node-level atom diffusion.
t_graph = torch.rand(dng_batch.num_graphs, device=device)
t_node = t_graph[dng_batch.batch]

# Simplified Algorithm 1: only diffuse lattice and atom features.
# We do not diffuse fractional coordinates or velocities in this notebook.
l_t, eps_l = diffusion.forward_sample(t_graph, dng_batch.l)
a_t, eps_a = diffusion.forward_sample(t_node, dng_batch.h)
target_l = diffusion.score_target(t_graph, eps_l)
target_a = diffusion.score_target(t_node, eps_a)

# CSPVNet still expects positions and a velocity-like tensor.
# Here we pass clean fractional coordinates and set v = 0.
v_t = torch.zeros_like(dng_batch.pos)

score_network = CSPVNet(
    hidden_dim=128,
    num_layers=4,
    h_dim=dng_batch.h.shape[-1],
    smooth=True,
    pred_v=True,
    pred_l=True,
    pred_h=True,
).to(device)

scores = score_network(
    t=t_graph[:, None],
    pos=dng_batch.pos,
    v=v_t,
    h=a_t,
    l=l_t,
    node_index=dng_batch.batch,
    edge_node_index=dng_batch.edge_node_index,
)

loss_l = denoise_score_matching_loss(scores['l'], target_l, t_graph, diffusion)
loss_a = denoise_score_matching_loss(scores['h'], target_a, t_node, diffusion)
loss_total = loss_l + loss_a

print('pos shape        :', tuple(dng_batch.pos.shape))
print('l_t shape        :', tuple(l_t.shape))
print('a_t shape        :', tuple(a_t.shape))
print('score_l shape    :', tuple(scores['l'].shape))
print('score_h shape    :', tuple(scores['h'].shape))
print('loss_l           :', float(loss_l))
print('loss_a           :', float(loss_a))
print('loss_total       :', float(loss_total))

# One tiny optimization step with the real score network.
score_optimizer = torch.optim.Adam(score_network.parameters(), lr=1e-3)
score_optimizer.zero_grad()
loss_total.backward()
score_optimizer.step()

pos shape        : (45, 3)
l_t shape        : (4, 6)
a_t shape        : (45, 118)
score_l shape    : (4, 6)
score_h shape    : (45, 118)
loss_l           : 0.5804222226142883
loss_a           : 1.1879991292953491
loss_total       : 1.7684214115142822


## 8. Simple Training Loop With The Real Score Network

This is the same idea as above, but now written as a very small training loop.

For each batch we:

- sample graph times `t_graph`
- expand them to node times `t_node`
- diffuse lattice and atom features only
- keep positions fixed and set velocity to zero
- predict lattice and atom scores with `CSPVNet`
- optimize the sum of the two DSM losses

In [89]:
train_loader = DNGTask().dataloader(root=root, split='train', batch_size=16, shuffle=True, download=True)
score_network = CSPVNet(
    hidden_dim=128,
    num_layers=4,
    h_dim=118,
    smooth=True,
    pred_v=True,
    pred_l=True,
    pred_h=True,
).to(device)
score_optimizer = torch.optim.Adam(score_network.parameters(), lr=1e-3)

num_epochs = 5
print_every = 20

for epoch in range(num_epochs):
    running_total = 0.0

    for step, batch in enumerate(train_loader):
        batch = batch.to(device)

        t_graph = torch.rand(batch.num_graphs, device=device)
        t_node = t_graph[batch.batch]

        # Simplified Algorithm 1 targets: only l_t and a_t are noised.
        l_t, eps_l = diffusion.forward_sample(t_graph, batch.l)
        a_t, eps_a = diffusion.forward_sample(t_node, batch.h)
        target_l = diffusion.score_target(t_graph, eps_l)
        target_a = diffusion.score_target(t_node, eps_a)

        scores = score_network(
            t=t_graph[:, None],
            pos=batch.pos,
            v=torch.zeros_like(batch.pos),
            h=a_t,
            l=l_t,
            node_index=batch.batch,
            edge_node_index=batch.edge_node_index,
        )

        loss_l = denoise_score_matching_loss(scores['l'], target_l, t_graph, diffusion)
        loss_a = denoise_score_matching_loss(scores['h'], target_a, t_node, diffusion)
        loss_total = loss_l + loss_a

        score_optimizer.zero_grad()
        loss_total.backward()
        score_optimizer.step()

        running_total += float(loss_total)

        if step % print_every == 0:
            print(
                f'epoch={epoch:02d} step={step:03d} '
                f'loss_total={float(loss_total):.6f} '
                f'loss_l={float(loss_l):.6f} '
                f'loss_a={float(loss_a):.6f}'
            )

    mean_total = running_total / (step + 1)
    print(f'epoch={epoch:02d} mean_total={mean_total:.6f}')

epoch=00 step=000 loss_total=2.113068 loss_l=0.983446 loss_a=1.129622
epoch=00 step=020 loss_total=1.888188 loss_l=0.906111 loss_a=0.982077
epoch=00 step=040 loss_total=2.090187 loss_l=1.191919 loss_a=0.898269
epoch=00 step=060 loss_total=1.528869 loss_l=0.677454 loss_a=0.851416
epoch=00 step=080 loss_total=1.825851 loss_l=1.067543 loss_a=0.758308
epoch=00 step=100 loss_total=1.256539 loss_l=0.589888 loss_a=0.666651
epoch=00 step=120 loss_total=0.872219 loss_l=0.272295 loss_a=0.599924
epoch=00 step=140 loss_total=0.858603 loss_l=0.309381 loss_a=0.549222
epoch=00 step=160 loss_total=0.724288 loss_l=0.220548 loss_a=0.503740
epoch=00 step=180 loss_total=0.746636 loss_l=0.247758 loss_a=0.498878
epoch=00 step=200 loss_total=0.707176 loss_l=0.217031 loss_a=0.490146
epoch=00 step=220 loss_total=0.569535 loss_l=0.172714 loss_a=0.396821
epoch=00 step=240 loss_total=0.422612 loss_l=0.099381 loss_a=0.323231
epoch=00 step=260 loss_total=0.353575 loss_l=0.066064 loss_a=0.287511
epoch=00 step=280 lo

KeyboardInterrupt: 